# 06 — RAG runs (Dense / BM25 / Hybrid)

Writes results into `RESULTS_DIR`.

In [54]:
def _count_tokens_with_model(llm, text: str) -> int:
    return len(llm.tokenize(text.encode("utf-8")))

def _truncate_to_tokens(llm, text: str, max_tokens: int) -> str:
    toks = llm.tokenize(text.encode("utf-8"))
    toks = toks[:max_tokens]
    return llm.detokenize(toks).decode("utf-8", errors="ignore")

def build_rag_prompt_budgeted(question, contexts, llm, max_input_tokens=512):
    system_part = (
        "You are a question answering assistant. "
        "Use ONLY the given context. "
        "Return ONLY the short answer phrase copied verbatim from the context. "
        "Do not add any explanation or extra words. "
        "If the answer is not in the context, say \"I don't know\"."
    )

    sep = "\n\n---\n\n"

    base_prompt = (
        system_part
        + "\n\nContext:\n\n"
        + f"Question: {question}\n"
        + "Answer:"
    )

    base_len = _count_tokens_with_model(llm, base_prompt)
    context_budget = max_input_tokens - base_len
    if context_budget <= 0:
        return base_prompt

    included = []
    used = 0

    for ctx in contexts:
        # include separator cost if this isn't the first chunk
        sep_cost = _count_tokens_with_model(llm, sep) if included else 0
        ctx_cost = _count_tokens_with_model(llm, ctx)

        if used + sep_cost + ctx_cost <= context_budget:
            included.append(ctx)
            used += sep_cost + ctx_cost
        else:
            # Try truncating this chunk to fit remaining budget
            remaining = context_budget - used - sep_cost
            if remaining > 0:
                truncated = _truncate_to_tokens(llm, ctx, remaining)
                if truncated.strip():
                    included.append(truncated)
                    used += sep_cost + _count_tokens_with_model(llm, truncated)
            # don't break; try next contexts
            continue

    final_prompt = (
        system_part
        + "\n\nContext:\n\n"
        + (sep.join(included) if included else "")
        + "\n\nQuestion: "
        + question
        + "\nAnswer:"
    )
    return final_prompt


In [55]:
import re

def clean_answer(raw_answer: str) -> str:
    """
    Post-process the raw model output to enforce a short phrase answer.
    Strategy:
      - Take the first non-empty line.
      - Strip trailing '.', '?', '!' if present.
      - Strip common prefixes ("The answer is", "Answer:", "Assistant:", etc.).
      - Strip surrounding quotes.
      - If the cleaned answer starts with 'Context', drop that word.
    """
    if not raw_answer:
        return raw_answer

    # 1) First non-empty line
    lines = [l.strip() for l in raw_answer.splitlines() if l.strip()]
    if not lines:
        return ""

    line = lines[0]
    ans = line.strip()

    # 2) Drop only trailing sentence-ending punctuation (keep the rest of the phrase)
    ans = ans.rstrip(".?!").strip()

    # 3) Remove common leading phrases (case-insensitive)
    lower = ans.lower()
    prefixes = [
        "the answer is",
        "the answer:",
        "answer is",
        "answer:",
        "it is",
        "it's",
        "it was",
        "the country is",
        "assistant:",
        "model:",
    ]
    for p in prefixes:
        if lower.startswith(p):
            ans = ans[len(p):].strip(" ,:-")
            break

    # 4) Strip surrounding quotes if present
    ans = ans.strip()
    if len(ans) >= 2 and (
        (ans[0] == ans[-1] == '"') or
        (ans[0] == ans[-1] == "'")
    ):
        ans = ans[1:-1].strip()

    # 5) If it starts with 'context', drop that token
    if ans.lower().startswith("context"):
        parts = ans.split()
        ans = " ".join(parts[1:]) if len(parts) > 1 else ""

    # 6) Final trim + collapse multiple spaces
    ans = re.sub(r"\s+", " ", ans).strip()

    return ans


In [56]:
def generate_answer_from_context(
    question: str,
    contexts: list[str],
    max_input_tokens: int = 512,
    max_new_tokens: int = 32,
) -> str:
   
    # 1) Build prompt with context budgeting (uses qwen_llm.tokenize internally)
    prompt = build_rag_prompt_budgeted(question, contexts, llm=qwen_llm, max_input_tokens=max_input_tokens)

    # 2) Call the llama-cpp model directly
    # n_ctx in qwen_llm must be >= max_input_tokens + max_new_tokens
    result = qwen_llm(
        prompt,
        max_tokens=max_new_tokens,   # number of new tokens to generate
        temperature=0.0,             # fully greedy / deterministic
        top_p=1.0,
    )

    # 3) Extract text and clean
    raw_answer = result["choices"][0]["text"].strip()
    return clean_answer(raw_answer)


In [57]:
import time
import psutil
import pandas as pd
from tqdm.auto import tqdm

TOP_K = 10
EF_SEARCH = 96

retriever = hybrid_retriever
process = psutil.Process(os.getpid())

rag_results = []

for i, row in tqdm(questions_df.iterrows(), total=len(questions_df)):
    question = row["question"]

    loop_start = time.perf_counter()
    mem_before_mb = process.memory_info().rss / (1024 ** 2)

    # --- 1) Retrieve ---
    hits_df, timers = retriever.timed_search(question, k=TOP_K, efsearch=EF_SEARCH)

    texts = hits_df["text"].tolist()
    context_texts = [
        " ".join(map(str, t)) if isinstance(t, list) else str(t)
        for t in texts
    ]
   

    # --- 2) Generate ---
    gen_start = time.perf_counter()
    gen_answer = generate_answer_from_context(
        question=question,
        contexts=context_texts,
        max_input_tokens=512,
        max_new_tokens=32
    )
    generation_latency_s = (time.perf_counter() - gen_start)
    generation_latency_ms = (time.perf_counter() - gen_start) * 1000

    # --- 3) Timings & memory ---
    total_latency_ms = (time.perf_counter() - loop_start) * 1000
    mem_after_mb = process.memory_info().rss / (1024 ** 2)
    mem_delta_mb = mem_after_mb - mem_before_mb

    gold_answer = row.get("answer_text", None)

    rag_results.append({
        "question_id": row.get("question_id", i),
        "question": question,
        "generated_answer": gen_answer,
        "gold_answer_text": gold_answer,
        "retrieved_passage_ids": hits_df["passage_id"].tolist() if "passage_id" in hits_df.columns else hits_df.index.tolist(),
        "retrieved_passages": context_texts,
        "generation_latency_ms": generation_latency_ms,
        "memory_rss_mb": mem_after_mb,
    })

Print("RAG Results for Hybrid:")

rag_results_df_hybrid = pd.DataFrame(rag_results)


  0%|          | 0/50 [00:00<?, ?it/s]

question_id  \
0   sfq_26014   

                                                                   question  \
0  The Gulf of Carpentaria is an inlet on the north coast of which country?   

     generated_answer  \
0  northern Australia   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    gold_answer_text  \
0  AUSTRALIA , AUSTRALIAN , Asutralia , Ausrtalia , Aussieland , Austalia , Austraila , Austrailia , Australai , AustraliA , Australia , Australia (Commonwealth realm) , Australia (Commonwealth) , Australia (commonwealth realm) , Australia (commonwealth) , Australia (constitutional monarchy) , Australia (country) , Australia (dominion) , Australia (empire) , Australia (federation) , Australia (monarchy) , Australia (nation state) , Australia (nation) , Australia (nation-state) , Australia (realm) , Australia (state) , Australia's , Australia. , Australian Commonwealth , Australian Woman's Day , Australian city life , Australian country life , Australian geopolitics , Australian mainland , Australian's , Australias , Australie , Australien , Australiia , Australija , Australlia , Australo , Australo- , Australocentric , Australocentrism , Australocentrist , Austraya , Austrlaia , Austrlia , City life in Australia , Commonwealth Australia , Commonwealth of Australia , Commonwealth of australia , Continental Australia , Country life in Australia , Dominion of Australia , Empire of Australia , Etymology of Australia , Federal Australia , Geopolitics of Australia , ISO 3166-1:AU , Imperial Australia , Mainland Australia , Modern Australia , New Australian , Orstraya , Ostralia , Pax Australiana , Peace of Australia , Philosophy in Australia , Science in Australia , Straya , Technology in Australia , The Commonwealth of Australia , United States of Australia   

                                                                                                                                                                                            retrieved_passage_ids  \
0  [sfq_26014::r417::c0, sfq_26014::r417::c2, sfq_24548::r355::c78, qb_4695::r279::c43, sfq_8782::r165::c7, qw_6834::r93::c19, bb_1870::r330::c13, sfq_7880::r300::c14, odql_6295::r248::c37, bb_1575::r440::c33]   

                                                                                                                                                                                                                                                                                                                                                                                 

In [58]:
import time
import psutil
import pandas as pd
from tqdm.auto import tqdm

TOP_K = 10
EF_SEARCH = 96

retriever = dense_retriever
process = psutil.Process(os.getpid())

rag_results = []

for i, row in tqdm(questions_df.iterrows(), total=len(questions_df)):
    question = row["question"]

    loop_start = time.perf_counter()
    mem_before_mb = process.memory_info().rss / (1024 ** 2)

    # --- 1) Retrieve ---
    hits_df, timers = retriever.timed_search(
        question,
        k=TOP_K,
        efsearch=EF_SEARCH
    )

    context_texts = hits_df["text"].apply(lambda x: " ".join(map(str, x)) if isinstance(x, list) else str(x)).tolist()
    

    # --- 2) Generate ---
    gen_start = time.perf_counter()
    gen_answer = generate_answer_from_context(
        question=question,
        contexts=context_texts,
        max_input_tokens=512,
        max_new_tokens=16
    )
    generation_latency_ms = (time.perf_counter() - gen_start) * 1000

    # --- 3) Timings & memory ---
    mem_after_mb = process.memory_info().rss / (1024 ** 2)
    mem_delta_mb = mem_after_mb - mem_before_mb

    gold_answer = row.get("answer_text", None)

    rag_results.append({
        "question_id": row.get("question_id", i),
        "question": question,
        "generated_answer": gen_answer,
        "gold_answer_text": gold_answer,
        "retrieved_passage_ids": hits_df["passage_id"].tolist() if "passage_id" in hits_df.columns else hits_df.index.tolist(),
        "retrieved_passages": context_texts,
        "generation_latency_ms": generation_latency_ms,
        "memory_rss_mb": mem_after_mb,
    })

Print("RAG Results for Dense:")
rag_results_df_dense = pd.DataFrame(rag_results)



  0%|          | 0/50 [00:00<?, ?it/s]

question_id  \
0   sfq_26014   

                                                                   question  \
0  The Gulf of Carpentaria is an inlet on the north coast of which country?   

     generated_answer  \
0  northern Australia   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    gold_answer_text  \
0  AUSTRALIA , AUSTRALIAN , Asutralia , Ausrtalia , Aussieland , Austalia , Austraila , Austrailia , Australai , AustraliA , Australia , Australia (Commonwealth realm) , Australia (Commonwealth) , Australia (commonwealth realm) , Australia (commonwealth) , Australia (constitutional monarchy) , Australia (country) , Australia (dominion) , Australia (empire) , Australia (federation) , Australia (monarchy) , Australia (nation state) , Australia (nation) , Australia (nation-state) , Australia (realm) , Australia (state) , Australia's , Australia. , Australian Commonwealth , Australian Woman's Day , Australian city life , Australian country life , Australian geopolitics , Australian mainland , Australian's , Australias , Australie , Australien , Australiia , Australija , Australlia , Australo , Australo- , Australocentric , Australocentrism , Australocentrist , Austraya , Austrlaia , Austrlia , City life in Australia , Commonwealth Australia , Commonwealth of Australia , Commonwealth of australia , Continental Australia , Country life in Australia , Dominion of Australia , Empire of Australia , Etymology of Australia , Federal Australia , Geopolitics of Australia , ISO 3166-1:AU , Imperial Australia , Mainland Australia , Modern Australia , New Australian , Orstraya , Ostralia , Pax Australiana , Peace of Australia , Philosophy in Australia , Science in Australia , Straya , Technology in Australia , The Commonwealth of Australia , United States of Australia   

                                                                                                                                                                                            retrieved_passage_ids  \
0  [sfq_26014::r417::c0, sfq_24548::r355::c78, sfq_26014::r417::c2, qb_4695::r279::c43, sfq_8782::r165::c7, qw_6834::r93::c19, bb_1870::r330::c13, sfq_7880::r300::c14, odql_6295::r248::c37, bb_1575::r440::c33]   

                                                                                                                                                                                                                                                                                                                                                                                 

In [59]:
import time
import psutil
import pandas as pd
from tqdm.auto import tqdm

TOP_K = 10
EF_SEARCH = 96

retriever = bm25_retriever
process = psutil.Process(os.getpid())

rag_results = []

for i, row in tqdm(questions_df.iterrows(), total=len(questions_df)):
    question = row["question"]

    loop_start = time.perf_counter()
    mem_before_mb = process.memory_info().rss / (1024 ** 2)

    # --- 1) Retrieve ---
    hits_df, timers = retriever.timed_search(
        question,
        k=TOP_K,
        efsearch=EF_SEARCH
    )

    context_texts = hits_df["text"].apply(lambda x: " ".join(map(str, x)) if isinstance(x, list) else str(x)).tolist()
    

    # --- 2) Generate ---
    gen_start = time.perf_counter()
    gen_answer = generate_answer_from_context(
        question=question,
        contexts=context_texts,
        max_input_tokens=512,
        max_new_tokens=16
    )
    generation_latency_ms = (time.perf_counter() - gen_start) * 1000

    # --- 3) Timings & memory ---
   
    mem_after_mb = process.memory_info().rss / (1024 ** 2)
    mem_delta_mb = mem_after_mb - mem_before_mb

    gold_answer = row.get("answer_text", None)

    rag_results.append({
        "question_id": row.get("question_id", i),
        "question": question,
        "generated_answer": gen_answer,
        "gold_answer_text": gold_answer,
        "retrieved_passage_ids": hits_df["passage_id"].tolist() if "passage_id" in hits_df.columns else hits_df.index.tolist(),
        "retrieved_passages": context_texts,
        "generation_latency_ms": generation_latency_ms,
        "memory_rss_mb": mem_after_mb,
    })

Print("RAG Results for BM25:")

rag_results_df_bm25 = pd.DataFrame(rag_results)


  0%|          | 0/50 [00:00<?, ?it/s]

question_id  \
0   sfq_26014   

                                                                   question  \
0  The Gulf of Carpentaria is an inlet on the north coast of which country?   

     generated_answer  \
0  northern Australia   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    gold_answer_text  \
0  AUSTRALIA , AUSTRALIAN , Asutralia , Ausrtalia , Aussieland , Austalia , Austraila , Austrailia , Australai , AustraliA , Australia , Australia (Commonwealth realm) , Australia (Commonwealth) , Australia (commonwealth realm) , Australia (commonwealth) , Australia (constitutional monarchy) , Australia (country) , Australia (dominion) , Australia (empire) , Australia (federation) , Australia (monarchy) , Australia (nation state) , Australia (nation) , Australia (nation-state) , Australia (realm) , Australia (state) , Australia's , Australia. , Australian Commonwealth , Australian Woman's Day , Australian city life , Australian country life , Australian geopolitics , Australian mainland , Australian's , Australias , Australie , Australien , Australiia , Australija , Australlia , Australo , Australo- , Australocentric , Australocentrism , Australocentrist , Austraya , Austrlaia , Austrlia , City life in Australia , Commonwealth Australia , Commonwealth of Australia , Commonwealth of australia , Continental Australia , Country life in Australia , Dominion of Australia , Empire of Australia , Etymology of Australia , Federal Australia , Geopolitics of Australia , ISO 3166-1:AU , Imperial Australia , Mainland Australia , Modern Australia , New Australian , Orstraya , Ostralia , Pax Australiana , Peace of Australia , Philosophy in Australia , Science in Australia , Straya , Technology in Australia , The Commonwealth of Australia , United States of Australia   

                                                                                                                                                                                                    retrieved_passage_ids  \
0  [sfq_26014::r417::c0, sfq_26014::r417::c2, sfq_24548::r355::c111, sfq_26014::r417::c1, sfq_24548::r355::c110, odql_6295::r248::c57, wh_431::r107::c11, sfq_24548::r355::c107, wh_431::r107::c10, odql_6295::r248::c74]   

                                                                                                                                                                                                                                                                                                                                                                 

In [ ]:
RAG_OUTPUT_PATH_DENSE = RESULTS_DIR / "rag_answers_dense.parquet"
RAG_OUTPUT_PATH_DENSE.parent.mkdir(parents=True, exist_ok=True)

RAG_OUTPUT_PATH_BM25 = RESULTS_DIR / "rag_answers_bm25.parquet"
RAG_OUTPUT_PATH_BM25.parent.mkdir(parents=True, exist_ok=True)

RAG_OUTPUT_PATH_HYBRID = RESULTS_DIR / "rag_answers_hybrid.parquet"
RAG_OUTPUT_PATH_HYBRID.parent.mkdir(parents=True, exist_ok=True)




score_cols = ["question_id", "question", "generated_answer", "gold_answer_text"]
score_cols = [c for c in score_cols if c in rag_results_df_hybrid.columns]
df_score_hybrid = rag_results_df_hybrid[score_cols].copy()
df_score_bm25 = rag_results_df_bm25[score_cols].copy()
df_score_dense = rag_results_df_dense[score_cols].copy()

df_score_dense.to_parquet(RAG_OUTPUT_PATH_DENSE, index=False)
print("Saved RAG outputs to:", RAG_OUTPUT_PATH_DENSE)

df_score_bm25.to_parquet(RAG_OUTPUT_PATH_BM25, index=False)
print("Saved RAG outputs to:", RAG_OUTPUT_PATH_BM25)
df_score_hybrid.to_parquet(RAG_OUTPUT_PATH_HYBRID, index=False)
print("Saved RAG outputs to:", RAG_OUTPUT_PATH_HYBRID)
